In [ ]:
'''
generating the data
'''

!pip install numpy pandas networkx matplotlib seaborn
!python generate_dtsp_data.py --num-stations 15 --network-type circular --output-dir data/dtsp_synthetic

In [ ]:
"""
DTSP Data Explorer - Jupyter Notebook Code

Run this after generating synthetic data to explore and visualize the dataset.
Copy this into a Jupyter notebook for interactive exploration.
"""

# ========================================
# CELL 1: Setup and Imports
# ========================================

import pandas as pd
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

# Data directory
DATA_DIR = Path('data/dtsp_synthetic')

print("DTSP Data Explorer")
print("=" * 60)


# ========================================
# CELL 2: Load Data
# ========================================

print("\nLoading dataset...")

# Load network
network = nx.read_gml(DATA_DIR / 'network.gml')
print(f"✓ Network loaded: {network.number_of_nodes()} stations, {network.number_of_edges()} tracks")

# Load stations
stations = pd.read_csv(DATA_DIR / 'stations.csv')
print(f"✓ Stations loaded: {len(stations)} rows")

# Load OD matrix
od_matrix = pd.read_csv(DATA_DIR / 'od_matrix.csv')
print(f"✓ OD matrix loaded: {len(od_matrix)} OD pairs")

# Load hourly demand
hourly_demand = pd.read_csv(DATA_DIR / 'hourly_demand.csv')
print(f"✓ Hourly demand loaded: {len(hourly_demand)} records")

# Load parameters
with open(DATA_DIR / 'operational_params.json') as f:
    params = json.load(f)
print(f"✓ Operational parameters loaded")

# Load summary
with open(DATA_DIR / 'dataset_summary.json') as f:
    summary = json.load(f)
print(f"✓ Summary loaded")

print("\n✓ All data loaded successfully!")

In [ ]:
# ========================================
# CELL 3: Network Visualization
# ========================================

print("\n" + "=" * 60)
print("NETWORK TOPOLOGY")
print("=" * 60)

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Extract node positions from coordinates
pos = {}
for node in network.nodes():
    coords_str = network.nodes[node]['coordinates']
    # Parse string representation of tuple
    coords = eval(coords_str) if isinstance(coords_str, str) else coords_str
    pos[node] = coords

# Plot 1: Basic network structure
ax1 = axes[0]
nx.draw_networkx_nodes(network, pos, node_size=300, node_color='lightblue', ax=ax1)
nx.draw_networkx_edges(network, pos, edge_color='gray', alpha=0.5, ax=ax1)
nx.draw_networkx_labels(network, pos, font_size=8, ax=ax1)
ax1.set_title('Network Topology', fontsize=14, fontweight='bold')
ax1.axis('off')

# Plot 2: Colored by station type
ax2 = axes[1]
transfer_stations = [n for n in network.nodes() if network.nodes[n]['is_transfer'] == 'True']
regular_stations = [n for n in network.nodes() if network.nodes[n]['is_transfer'] == 'False']

nx.draw_networkx_nodes(network, pos, nodelist=transfer_stations, 
                       node_size=500, node_color='red', 
                       label='Transfer Station', ax=ax2)
nx.draw_networkx_nodes(network, pos, nodelist=regular_stations,
                       node_size=300, node_color='lightblue',
                       label='Regular Station', ax=ax2)
nx.draw_networkx_edges(network, pos, edge_color='gray', alpha=0.5, ax=ax2)
nx.draw_networkx_labels(network, pos, font_size=8, ax=ax2)
ax2.set_title('Station Types', fontsize=14, fontweight='bold')
ax2.legend(loc='upper right')
ax2.axis('off')

plt.tight_layout()
plt.show()

# Network statistics
print(f"\nNetwork Statistics:")
print(f"  Total stations: {network.number_of_nodes()}")
print(f"  Total tracks: {network.number_of_edges()}")
print(f"  Transfer stations: {len(transfer_stations)}")
print(f"  Total network length: {summary['network']['total_length_km']} km")
print(f"  Avg inter-station distance: {summary['network']['avg_inter_station_distance']} km")



In [ ]:
# ========================================
# CELL 4: Station Analysis
# ========================================

print("\n" + "=" * 60)
print("STATION CHARACTERISTICS")
print("=" * 60)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Platform capacity distribution
ax1 = axes[0, 0]
stations['platform_capacity'].hist(bins=20, ax=ax1, color='skyblue', edgecolor='black')
ax1.set_xlabel('Platform Capacity (passengers)', fontsize=11)
ax1.set_ylabel('Number of Stations', fontsize=11)
ax1.set_title('Platform Capacity Distribution', fontsize=12, fontweight='bold')
ax1.axvline(stations['platform_capacity'].mean(), color='red', 
            linestyle='--', label=f"Mean: {stations['platform_capacity'].mean():.0f}")
ax1.legend()

# Plot 2: Zone distribution
ax2 = axes[0, 1]
zone_counts = stations['zone'].value_counts()
ax2.pie(zone_counts.values, labels=zone_counts.index, autopct='%1.1f%%',
        colors=['#ff9999', '#66b3ff'], startangle=90)
ax2.set_title('Station Zones', fontsize=12, fontweight='bold')

# Plot 3: Transfer vs Regular
ax3 = axes[1, 0]
transfer_counts = stations['is_transfer'].value_counts()
transfer_counts.index = ['Regular', 'Transfer'] if False in transfer_counts.index else ['Transfer', 'Regular']
ax3.bar(transfer_counts.index, transfer_counts.values, color=['lightblue', 'coral'])
ax3.set_ylabel('Number of Stations', fontsize=11)
ax3.set_title('Transfer Station Distribution', fontsize=12, fontweight='bold')
for i, v in enumerate(transfer_counts.values):
    ax3.text(i, v + 0.2, str(v), ha='center', fontweight='bold')

# Plot 4: Base demand multiplier
ax4 = axes[1, 1]
stations['base_demand_multiplier'].hist(bins=15, ax=ax4, color='lightgreen', edgecolor='black')
ax4.set_xlabel('Demand Multiplier', fontsize=11)
ax4.set_ylabel('Number of Stations', fontsize=11)
ax4.set_title('Station Demand Multipliers', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

# Top stations by capacity
print("\nTop 5 stations by platform capacity:")
print(stations.nlargest(5, 'platform_capacity')[['name', 'platform_capacity', 'is_transfer', 'zone']])

In [ ]:
# ========================================
# CELL 5: Demand Analysis
# ========================================

print("\n" + "=" * 60)
print("DEMAND PATTERNS")
print("=" * 60)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Demand by time period
ax1 = axes[0, 0]
period_demand = od_matrix.groupby('time_period')['avg_demand'].sum().sort_values(ascending=False)
colors = plt.cm.Set3(range(len(period_demand)))
ax1.bar(period_demand.index, period_demand.values, color=colors)
ax1.set_ylabel('Total Passengers', fontsize=11)
ax1.set_xlabel('Time Period', fontsize=11)
ax1.set_title('Demand by Time Period', fontsize=12, fontweight='bold')
ax1.tick_params(axis='x', rotation=45)
for i, v in enumerate(period_demand.values):
    ax1.text(i, v + 100, f'{v:,.0f}', ha='center', fontsize=9)

# Plot 2: Hourly demand profile
ax2 = axes[0, 1]
hourly_total = hourly_demand.groupby('hour')['demand'].sum()
ax2.plot(hourly_total.index, hourly_total.values, marker='o', linewidth=2, color='darkblue')
ax2.fill_between(hourly_total.index, hourly_total.values, alpha=0.3)
ax2.set_xlabel('Hour of Day', fontsize=11)
ax2.set_ylabel('Total Passengers', fontsize=11)
ax2.set_title('Hourly Demand Profile', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.set_xticks(range(0, 24, 2))

# Highlight peak hours
morning_peak = hourly_total[7:9].mean()
evening_peak = hourly_total[17:20].mean()
ax2.axvspan(7, 9, alpha=0.2, color='red', label='Morning Peak')
ax2.axvspan(17, 20, alpha=0.2, color='orange', label='Evening Peak')
ax2.legend()

# Plot 3: Top OD pairs
ax3 = axes[1, 0]
top_od = od_matrix.nlargest(10, 'avg_demand')[['origin', 'destination', 'avg_demand', 'time_period']]
od_labels = [f"{row['origin']}→{row['destination']}\n({row['time_period']})" 
             for _, row in top_od.iterrows()]
ax3.barh(range(len(top_od)), top_od['avg_demand'].values, color='teal')
ax3.set_yticks(range(len(top_od)))
ax3.set_yticklabels(od_labels, fontsize=8)
ax3.set_xlabel('Average Demand (passengers)', fontsize=11)
ax3.set_title('Top 10 OD Pairs', fontsize=12, fontweight='bold')
ax3.invert_yaxis()

# Plot 4: Demand distribution
ax4 = axes[1, 1]
ax4.hist(od_matrix['avg_demand'], bins=50, color='purple', alpha=0.7, edgecolor='black')
ax4.set_xlabel('Average Demand (passengers)', fontsize=11)
ax4.set_ylabel('Number of OD Pairs', fontsize=11)
ax4.set_title('Demand Distribution Across OD Pairs', fontsize=12, fontweight='bold')
ax4.axvline(od_matrix['avg_demand'].median(), color='red', 
            linestyle='--', label=f"Median: {od_matrix['avg_demand'].median():.1f}")
ax4.legend()

plt.tight_layout()
plt.show()

# Demand statistics
print("\nDemand Statistics:")
print(f"  Total daily passengers: {summary['demand']['total_daily_passengers']:,.0f}")
print(f"  Average per OD pair: {od_matrix['avg_demand'].mean():.2f}")
print(f"  Median per OD pair: {od_matrix['avg_demand'].median():.2f}")
print(f"  Peak period: {summary['demand']['peak_demand_period']}")

In [ ]:
# ========================================
# CELL 6: Origin-Destination Heatmap
# ========================================

print("\n" + "=" * 60)
print("OD MATRIX HEATMAP (Morning Peak)")
print("=" * 60)

# Create OD matrix for morning peak
morning_peak_od = od_matrix[od_matrix['time_period'] == 'morning_peak'].copy()
od_pivot = morning_peak_od.pivot_table(
    index='origin', 
    columns='destination', 
    values='avg_demand',
    fill_value=0
)

# Plot heatmap
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(od_pivot, annot=False, cmap='YlOrRd', ax=ax, cbar_kws={'label': 'Passengers'})
ax.set_title('Morning Peak OD Matrix', fontsize=14, fontweight='bold')
ax.set_xlabel('Destination Station', fontsize=12)
ax.set_ylabel('Origin Station', fontsize=12)
plt.tight_layout()
plt.show()

print("\nHeatmap shows demand flow during morning peak (7-9am)")
print("Darker colors = higher passenger volume")

In [ ]:
# ========================================
# CELL 7: Track Characteristics
# ========================================

print("\n" + "=" * 60)
print("TRACK CHARACTERISTICS")
print("=" * 60)

# Extract edge data
edge_data = []
for origin, dest, data in network.edges(data=True):
    edge_data.append({
        'origin': origin,
        'destination': dest,
        'distance_km': float(data['distance_km']),
        'avg_travel_time': float(data['avg_travel_time']),
        'capacity': int(data['capacity_trains_per_hour']),
        'reliability': float(data['reliability']),
        'delay_std': float(data['delay_std'])
    })

tracks_df = pd.DataFrame(edge_data)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Travel time distribution
ax1 = axes[0, 0]
tracks_df['avg_travel_time'].hist(bins=20, ax=ax1, color='lightcoral', edgecolor='black')
ax1.set_xlabel('Travel Time (minutes)', fontsize=11)
ax1.set_ylabel('Number of Tracks', fontsize=11)
ax1.set_title('Travel Time Distribution', fontsize=12, fontweight='bold')
ax1.axvline(tracks_df['avg_travel_time'].mean(), color='red', 
            linestyle='--', label=f"Mean: {tracks_df['avg_travel_time'].mean():.2f} min")
ax1.legend()

# Plot 2: Track capacity
ax2 = axes[0, 1]
tracks_df['capacity'].hist(bins=15, ax=ax2, color='lightgreen', edgecolor='black')
ax2.set_xlabel('Capacity (trains/hour)', fontsize=11)
ax2.set_ylabel('Number of Tracks', fontsize=11)
ax2.set_title('Track Capacity Distribution', fontsize=12, fontweight='bold')

# Plot 3: Reliability
ax3 = axes[1, 0]
tracks_df['reliability'].hist(bins=20, ax=ax3, color='skyblue', edgecolor='black')
ax3.set_xlabel('Reliability', fontsize=11)
ax3.set_ylabel('Number of Tracks', fontsize=11)
ax3.set_title('Track Reliability Distribution', fontsize=12, fontweight='bold')
ax3.axvline(tracks_df['reliability'].mean(), color='red', 
            linestyle='--', label=f"Mean: {tracks_df['reliability'].mean():.3f}")
ax3.legend()

# Plot 4: Distance vs Travel Time
ax4 = axes[1, 1]
ax4.scatter(tracks_df['distance_km'], tracks_df['avg_travel_time'], 
           alpha=0.6, s=100, color='purple')
ax4.set_xlabel('Distance (km)', fontsize=11)
ax4.set_ylabel('Travel Time (minutes)', fontsize=11)
ax4.set_title('Distance vs Travel Time', fontsize=12, fontweight='bold')
ax4.grid(True, alpha=0.3)

# Add trend line
z = np.polyfit(tracks_df['distance_km'], tracks_df['avg_travel_time'], 1)
p = np.poly1d(z)
x_trend = np.linspace(tracks_df['distance_km'].min(), tracks_df['distance_km'].max(), 100)
ax4.plot(x_trend, p(x_trend), "r--", alpha=0.8, label=f'y={z[0]:.2f}x+{z[1]:.2f}')
ax4.legend()

plt.tight_layout()
plt.show()

print("\nTrack Statistics:")
print(f"  Average travel time: {tracks_df['avg_travel_time'].mean():.2f} minutes")
print(f"  Average capacity: {tracks_df['capacity'].mean():.1f} trains/hour")
print(f"  Average reliability: {tracks_df['reliability'].mean():.3f}")
print(f"  Average distance: {tracks_df['distance_km'].mean():.2f} km")

In [ ]:
# ========================================
# CELL 8: Operational Parameters Summary
# ========================================

print("\n" + "=" * 60)
print("OPERATIONAL PARAMETERS")
print("=" * 60)

print("\nNetwork Constraints:")
for key, value in params['network'].items():
    print(f"  {key}: {value}")

print("\nTrain Specifications:")
for key, value in params['trains'].items():
    print(f"  {key}: {value}")

print("\nOperational Rules:")
for key, value in params['operations'].items():
    print(f"  {key}: {value}")

print("\nCost Parameters:")
for key, value in params['costs'].items():
    print(f"  {key}: {value}")

print("\nService Level Targets:")
for key, value in params['service_levels'].items():
    print(f"  {key}: {value}")

In [ ]:
# ========================================
# CELL 9: Quick Data Access Examples
# ========================================

print("\n" + "=" * 60)
print("QUICK DATA ACCESS EXAMPLES")
print("=" * 60)

print("\n# Get demand from station S00 to S05 during morning peak:")
example_demand = od_matrix[
    (od_matrix['origin'] == 'S00') & 
    (od_matrix['destination'] == 'S05') & 
    (od_matrix['time_period'] == 'morning_peak')
]
print(example_demand[['origin', 'destination', 'time_period', 'avg_demand']])

print("\n# Get all hourly demand for hour 8 (8am):")
hour_8_demand = hourly_demand[hourly_demand['hour'] == 8].head()
print(hour_8_demand[['origin', 'destination', 'hour', 'demand']])

print("\n# Get station information for S00:")
station_s00 = stations[stations['id'] == 'S00']
print(station_s00)

print("\n# Get all tracks connected to S00:")
s00_tracks = tracks_df[
    (tracks_df['origin'] == 'S00') | 
    (tracks_df['destination'] == 'S00')
]
print(s00_tracks[['origin', 'destination', 'distance_km', 'avg_travel_time']])

In [ ]:
# ========================================
# CELL 10: Data Quality Checks
# ========================================

print("\n" + "=" * 60)
print("DATA QUALITY CHECKS")
print("=" * 60)

checks_passed = 0
total_checks = 0

# Check 1: No missing values in OD matrix
total_checks += 1
if od_matrix.isnull().sum().sum() == 0:
    print("✓ No missing values in OD matrix")
    checks_passed += 1
else:
    print("✗ Missing values found in OD matrix")

# Check 2: All stations in OD matrix exist in network
total_checks += 1
od_stations = set(od_matrix['origin'].unique()) | set(od_matrix['destination'].unique())
network_stations = set(network.nodes())
if od_stations.issubset(network_stations):
    print("✓ All OD stations exist in network")
    checks_passed += 1
else:
    print("✗ Some OD stations not in network")

# Check 3: Demand values are non-negative
total_checks += 1
if (od_matrix['avg_demand'] >= 0).all() and (hourly_demand['demand'] >= 0).all():
    print("✓ All demand values are non-negative")
    checks_passed += 1
else:
    print("✗ Negative demand values found")

# Check 4: Travel times are reasonable
total_checks += 1
if (tracks_df['avg_travel_time'] > 0).all() and (tracks_df['avg_travel_time'] < 30).all():
    print("✓ Travel times are within reasonable range (0-30 minutes)")
    checks_passed += 1
else:
    print("✗ Unreasonable travel times detected")

# Check 5: Network is connected
total_checks += 1
if nx.is_weakly_connected(network):
    print("✓ Network is connected (all stations reachable)")
    checks_passed += 1
else:
    print("✗ Network has disconnected components")

print(f"\n{'='*60}")
print(f"Quality checks: {checks_passed}/{total_checks} passed")
print("✓ Dataset is ready for DTSP simulation!" if checks_passed == total_checks else "⚠ Review failed checks")

print("\n" + "=" * 60)
print("EXPLORATION COMPLETE")
print("=" * 60)
print("\nNext steps:")
print("1. Build simulation using this data")
print("2. Implement fitness function")
print("3. Create seed strategies")
print("4. Run ShinkaEvolve optimization")